In [9]:
import pandas as pd
import numpy as np
import re

from nltk import word_tokenize

import warnings
warnings.filterwarnings("ignore")

seed = 123

In [2]:
# load dataset
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

# rename target col to labels
train = train.rename(columns={"target": "labels"})
test["labels"] = 0

In [3]:
# clean text, store cleaned as col, "text_cleaned"
def clean_text(text_in:str):
    out = text_in
    out = re.sub("\n", " ", out) # strip multiline

    out = re.sub("B1G", "BIG", out) # strip multiline

    out = re.sub(r"http[^\s]+", " ", out) # strip http* link
    out = re.sub(r'--&gt;', '', out)
    out = out.replace(r'&lt;', '')
    out = out.replace(r'&gt;', '')
    out = out.replace(r'&amp;', '')
    out = re.sub(r"@\w+", "@USER", out) # @___ -> @USER
    out = re.sub(r"[" + re.escape('!"$%&()*+,-./:;<=>?[\\]^_`{|}~') + "]", " ", out) # most punctuation -> space
    out = re.sub(r"\d+", " ", out) # strip numbers
    out = re.sub(" ' ", " ", out) # most punctuation -> space

    # final space processing
    out = re.sub(r"\s+", " ", out) # multispace to 1 space
    out = re.sub(r"^\s+", "", out) # trim leading space
    out = re.sub(r"\s+$", "", out) # trim trailing space

    return out

train["text_cleaned"] = train["text"].apply(clean_text)
test["text_cleaned"] = test["text"].apply(clean_text)

# print before and after
_train_index = None
_test_index = None

# _train_index = 5803
# _test_index = 2125

_tmp_train = train.sample(1) if _train_index is None else train.loc[[_train_index]]
print("[Train] Row Index: ", _tmp_train.index[0])
print("[Train] Before clean: ", _tmp_train["text"].values[0])
print("[Train] After clean: ", _tmp_train["text_cleaned"].values[0])
print("===========================================================")
_tmp_test = test.sample(1) if _test_index is None else test.loc[[_test_index]]
print("[Test] Row Index: ", _tmp_test.index[0])
print("[Test] Before clean: ", _tmp_test["text"].values[0])
print("[Test] After clean: ", _tmp_test["text_cleaned"].values[0])

# clean tmp var
del _tmp_train, _tmp_test, _train_index, _test_index

[Train] Row Index:  6527
[Train] Before clean:  #Autoinsurance industry clueless on driverless cars : #healthinsurance http://t.co/YdEtWgRibk
[Train] After clean:  #Autoinsurance industry clueless on driverless cars #healthinsurance
[Test] Row Index:  1607
[Test] Before clean:  Good Day Columbia: First Responders Memorial with Sallie ... 
#MEMORIAL_DAY #MEMORIALDAY
 http://t.co/qW3uNfV8Kb http://t.co/hgxMBmrDa3
[Test] After clean:  Good Day Columbia First Responders Memorial with Sallie #MEMORIAL DAY #MEMORIALDAY


In [ ]:
# new feature
def count_pattern(pattern:str, text:str):
    return len(re.findall(pattern, text))

for df in [train, test]:
    df["mention_count"] = df["text"].apply(lambda x: count_pattern("@\w+", x))
    df["link_count"] = df["text"].apply(lambda x: count_pattern(r"http[^\s]+", x))
    df["exclamation_count"] = df["text"].apply(lambda x: count_pattern(r"!", x))
    df["char_count"] = df["text_cleaned"].apply(len)
    df["word_count"] = df["text_cleaned"].apply(lambda x: len(word_tokenize(x)))
    df["hashtag_count"] = df["text"].apply(lambda x: count_pattern(r"#\w+", x))
    df["uppercase_count"] = df["text"].apply(lambda x: len(re.findall(r"[A-Z]", x)))
    df["lowercase_count"] = df["text"].apply(lambda x: len(re.findall(r"[a-z]", x)))
    df["upper-to-all-ratio"] = df.apply(lambda x: x["uppercase_count"] / x["char_count"] if x["char_count"] > 0 else 0, axis=1)
    df["lower-to-all-ratio"] = df.apply(lambda x: x["lowercase_count"] / x["char_count"] if x["char_count"] > 0 else 0, axis=1)